# 02 — SQL Analysis (10 questions)

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/tmdb.db")

### Q1 — Action movies released after 2015

In [2]:
pd.read_sql("""
SELECT m.title, m.release_date
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g       ON mg.genre_id = g.genre_id
WHERE g.genre_name = 'Action' AND m.release_date >= '2016-01-01'
ORDER BY m.release_date;
""", conn)

,title,release_date
0,The 5th Wave,2016-01-14 00:00:00
1,Kung Fu Panda 3,2016-01-23 00:00:00
2,Zoolander 2,2016-02-06 00:00:00
3,Deadpool,2016-02-09 00:00:00
4,Triple 9,2016-02-19 00:00:00
...,...,...
263,F1,2025-06-25 00:00:00
264,Superman,2025-07-09 00:00:00
265,The Fantastic 4: First Steps,2025-07-23 00:00:00
266,Predator: Badlands,2025-11-05 00:00:00


### Q2 — Top 10 highest-grossing movies with their genres

In [3]:
pd.read_sql("""
SELECT m.title, m.revenue, GROUP_CONCAT(g.genre_name, ', ') AS genres
FROM movies m
LEFT JOIN movie_genres mg ON m.movie_id = mg.movie_id
LEFT JOIN genres g       ON mg.genre_id = g.genre_id
GROUP BY m.movie_id, m.title, m.revenue
ORDER BY m.revenue DESC
LIMIT 10;
""", conn)

,title,revenue,genres
0,Avatar,2923706026,"Adventure, Action, Science Fiction"
1,Avengers: Endgame,2799439100,"Adventure, Action, Science Fiction"
2,Avatar: The Way of Water,2334484620,"Adventure, Action, Science Fiction"
3,Titanic,2264162353,"Drama, Romance"
4,Star Wars: The Force Awakens,2068223624,"Adventure, Action, Science Fiction"
5,Avengers: Infinity War,2052415039,"Adventure, Action, Science Fiction"
6,Spider-Man: No Way Home,1921206586,"Adventure, Action, Science Fiction"
7,Zootopia 2,1868208796,"Adventure, Animation, Comedy, Mystery, Family"
8,Inside Out 2,1698863816,"Adventure, Animation, Comedy, Family"
9,Jurassic World,1671537444,"Adventure, Thriller, Science Fiction"


### Q3 — Average budget & revenue per genre

In [4]:
pd.read_sql("""
SELECT g.genre_name,
       AVG(m.budget)  AS avg_budget,
       AVG(m.revenue) AS avg_revenue
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g       ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
ORDER BY avg_revenue DESC;
""", conn)

,genre_name,avg_budget,avg_revenue
0,Adventure,1.093600e+08,3.902145e+08
1,Animation,8.336061e+07,3.597435e+08
2,Family,8.458426e+07,3.430968e+08
3,Science Fiction,9.525992e+07,3.235601e+08
4,Action,9.329457e+07,3.000547e+08
5,Fantasy,8.617784e+07,2.942975e+08
6,Comedy,5.256907e+07,2.113513e+08
7,Music,4.106939e+07,1.974526e+08
8,Romance,3.461113e+07,1.627896e+08
9,Thriller,4.938770e+07,1.605152e+08


### Q4 — Top 10 actors by number of movies

In [5]:
pd.read_sql("""
SELECT actor_name, COUNT(DISTINCT movie_id) AS movie_count
FROM cast
GROUP BY actor_name
ORDER BY movie_count DESC
LIMIT 10;
""", conn)

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Robert De Niro,38
2,Brad Pitt,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Willem Dafoe,32
6,Scarlett Johansson,32
7,Mark Wahlberg,32
8,Tom Cruise,31
9,Morgan Freeman,31


### Q5 — Directors with more than 3 movies

In [6]:
pd.read_sql("""
SELECT person_name, COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE job = 'Director'
GROUP BY person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC;
""", conn)

,person_name,movie_count
0,Steven Spielberg,27
1,Tim Burton,19
2,Ridley Scott,19
3,Martin Scorsese,16
4,Robert Zemeckis,15
...,...,...
212,Andrew Adamson,4
213,Alfonso Cuarón,4
214,Alexandre Aja,4
215,Adam Wingard,4


### Q6 — Top 10 most used keywords

In [7]:
pd.read_sql("""
SELECT keyword_id, COUNT(*) AS usage_count
FROM movie_keywords
GROUP BY keyword_id
ORDER BY usage_count DESC
LIMIT 10;
""", conn)

,keyword_id,usage_count
0,818,449
1,9663,416
2,179431,296
3,179430,242
4,9826,176
5,9672,166
6,242,161
7,3289,160
8,278823,155
9,9717,154


### Q7 — Movies with above-average budget but below-average revenue (subqueries)

In [8]:
pd.read_sql("""
SELECT m.title, m.budget, m.revenue
FROM movies m
WHERE m.budget > (SELECT AVG(budget) FROM movies)
  AND m.revenue < (SELECT AVG(revenue) FROM movies)
ORDER BY m.budget DESC;
""", conn)

,title,budget,revenue
0,The Marvels,274800000.0,206136825.0
1,Red One,250000000.0,185700759.0
2,Wake Up Dead Man: A Knives Out Mystery,210000000.0,4000000.0
3,Mulan,200000000.0,69965374.0
4,Wonder Woman 1984,200000000.0,169601036.0
...,...,...,...
312,Nightmare Alley,60000000.0,39629195.0
313,Damsel,60000000.0,5000.0
314,A Haunting in Venice,60000000.0,121400000.0
315,Gran Turismo,60000000.0,121700000.0


### Q8 — Actors who appeared in a Christopher Nolan film

In [9]:
pd.read_sql("""
SELECT DISTINCT c.actor_name
FROM cast c
JOIN crew cr ON c.movie_id = cr.movie_id
WHERE cr.person_name = 'Christopher Nolan' AND cr.job = 'Director'
ORDER BY c.actor_name;
""", conn)

,actor_name
0,Aaron Eckhart
1,Al Pacino
2,Alon Aboutboul
3,Andy Serkis
4,Aneurin Barnard
...,...
79,Tom Hardy
80,Tom Wilkinson
81,Topher Grace
82,Wes Bentley


### Q9 — Genre with highest average rating (min 20 movies)

In [10]:
pd.read_sql("""
SELECT g.genre_name,
       AVG(m.vote_average) AS avg_rating,
       COUNT(*) AS movie_count
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g       ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
HAVING COUNT(*) >= 20
ORDER BY avg_rating DESC
LIMIT 1;
""", conn)

,genre_name,avg_rating,movie_count
0,War,7.357576,85


### Q10 — Top 10 movies by number of keywords

In [11]:
pd.read_sql("""
SELECT m.title, COUNT(mk.keyword_id) AS keyword_count
FROM movies m
JOIN movie_keywords mk ON m.movie_id = mk.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC
LIMIT 10;
""", conn)

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,How to Train Your Dragon: The Hidden World,49
9,Pacific Rim: Uprising,49
